In [ ]:
!pip install -q ultralytics

In [ ]:
import os
import sys
import glob
import time
import json
import yaml
import random
import shutil
import warnings
import collections
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report
warnings.filterwarnings("ignore")

In [ ]:
class Config:
    SEED = 42
    EPOCHS = 5
    BATCH_SIZE = 16
    IMG_SIZE = 416
    WORKERS = min(4, os.cpu_count() or 1)
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 5e-4
    MIXED_PRECISION = torch.cuda.is_available()
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    BASE_DIR = "/kaggle/input/datasets/gurimakhija/ricedata"
    if not os.path.exists(BASE_DIR):
        _alt = glob.glob("/kaggle/input/*pest*")
        BASE_DIR = _alt[0] if _alt else "./rice-pest-detection"
        
    OUT_DIR = "./output/phase_1_2"
    YOLO_PROJ_DIR = os.path.join(OUT_DIR, "yolo_train")
    PLOT_DIR = os.path.join(OUT_DIR, "plots")
    
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_everything(Config.SEED)

In [ ]:
data_yaml_path = os.path.join(Config.BASE_DIR, "data.yaml")
if not os.path.exists(data_yaml_path):
    data_yaml_path = glob.glob(os.path.join(Config.BASE_DIR, "*.yaml"))
    data_yaml_path = data_yaml_path[0] if data_yaml_path else None

if not data_yaml_path:
    sys.exit(f"Invalid Dataset: data.yaml not found in {Config.BASE_DIR}. Halting execution.")

with open(data_yaml_path, 'r') as f:
    yaml_content = yaml.safe_load(f)

train_path = yaml_content["train"].replace("../", "")
val_path = yaml_content["val"].replace("../", "")

train_img_dir = os.path.join(Config.BASE_DIR, train_path)
val_img_dir = os.path.join(Config.BASE_DIR, val_path)

if not os.path.exists(train_img_dir) or not os.path.exists(val_img_dir):
    sys.exit("Invalid Dataset: Training or Validation image directories missing. Halting execution.")

class_names = yaml_content.get('names', [])
num_classes = yaml_content.get('nc', len(class_names))

train_images = glob.glob(os.path.join(train_img_dir, "*.*"))
val_images = glob.glob(os.path.join(val_img_dir, "*.*"))

train_labels_dir = train_img_dir.replace("images", "labels")
val_labels_dir = val_img_dir.replace("images", "labels")

train_labels = glob.glob(os.path.join(train_labels_dir, "*.txt"))

verification_report = {
    "total_train_images": len(train_images),
    "total_val_images": len(val_images),
    "total_train_labels": len(train_labels),
    "num_classes": num_classes,
    "class_names": class_names,
    "corrupted_images": 0,
    "empty_or_invalid_labels": 0
}

valid_train_images = []
for img_path in train_images:
    img = cv2.imread(img_path)
    if img is None:
        verification_report["corrupted_images"] += 1
        continue
    
    label_path = img_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"
    if not os.path.exists(label_path) or os.path.getsize(label_path) == 0:
        verification_report["empty_or_invalid_labels"] += 1
        continue
        
    with open(label_path, 'r') as f:
        valid_file = True
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                valid_file = False
                break
            try:
                cls_id = int(parts[0])
                if cls_id < 0 or cls_id >= num_classes:
                    valid_file = False
                    break
            except ValueError:
                valid_file = False
                break
        if not valid_file:
            verification_report["empty_or_invalid_labels"] += 1
            continue
            
    valid_train_images.append(img_path)

with open(os.path.join(Config.OUT_DIR, "dataset_verification.json"), "w") as f:
    json.dump(verification_report, f, indent=4)

if len(train_images) == 0 or len(train_labels) == 0:
    sys.exit("Dataset verification failed: Empty image or label directories. Halting execution.")


In [ ]:
def perform_yolo_eda(labels_dir, class_names):
    sns.set_theme(style="whitegrid")
    
    label_files = glob.glob(os.path.join(labels_dir, "*.txt"))
    all_classes = []
    bboxes = []
    objects_per_image = []
    
    for file in label_files:
        with open(file, 'r') as f:
            lines = f.readlines()
            objects_per_image.append(len(lines))
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    x_c, y_c, w, h = map(float, parts[1:])
                    all_classes.append(cls_id)
                    bboxes.append([x_c, y_c, w, h, w*h])
                    
    df_labels = pd.DataFrame(bboxes, columns=['x_center', 'y_center', 'width', 'height', 'area'])
    df_labels['class_id'] = all_classes
    df_labels['class_name'] = df_labels['class_id'].map(lambda x: class_names[x] if x < len(class_names) else "Unknown")
    
    def categorize_size(area):
        if area < 0.01: return 'Small'
        elif area < 0.1: return 'Medium'
        else: return 'Large'
        
    df_labels['bbox_size'] = df_labels['area'].apply(categorize_size)
    
    plt.figure(figsize=(14, 6))
    sns.countplot(data=df_labels, y='class_name', order=df_labels['class_name'].value_counts().index, palette='viridis')
    plt.title("Object Class Distribution in Training Set")
    plt.xlabel("Count")
    plt.ylabel("Pest Class")
    plt.tight_layout()
    plt.savefig(os.path.join(Config.PLOT_DIR, "class_distribution.png"), dpi=300)
    plt.show()
    plt.close()
    
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_labels, x='width', y='height', hue='class_name', alpha=0.5, palette='tab10', s=10)
    plt.title("Bounding Box Dimensions (Normalized Width vs Height)")
    plt.xlabel("Width")
    plt.ylabel("Height")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
    plt.tight_layout()
    plt.savefig(os.path.join(Config.PLOT_DIR, "bbox_dimensions.png"), dpi=300)
    plt.show()
    plt.close()
    
    plt.figure(figsize=(8, 8))
    sns.histplot(data=df_labels, x='x_center', y='y_center', bins=50, pmax=0.9, cmap='magma', cbar=True)
    plt.title("Spatial Heatmap of Bounding Box Centers")
    plt.xlim(0, 1)
    plt.ylim(1, 0)
    plt.tight_layout()
    plt.savefig(os.path.join(Config.PLOT_DIR, "bbox_spatial_heatmap.png"), dpi=300)
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 6))
    sns.countplot(data=df_labels, x='bbox_size', order=['Small', 'Medium', 'Large'], palette='Set2')
    plt.title("Bounding Box Size Distribution")
    plt.xlabel("Size Category")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(Config.PLOT_DIR, "bbox_size_distribution.png"), dpi=300)
    plt.show()
    plt.close()
    
    plt.figure(figsize=(8, 6))
    sns.histplot(objects_per_image, bins=range(1, max(objects_per_image)+2), discrete=True, color='teal')
    plt.title("Objects per Image Distribution")
    plt.xlabel("Number of Objects")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(os.path.join(Config.PLOT_DIR, "objects_per_image.png"), dpi=300)
    plt.show()
    plt.close()

perform_yolo_eda(train_labels_dir, class_names)

In [ ]:
dynamic_yaml_path = os.path.join(Config.OUT_DIR, "dynamic_data.yaml")
yaml_content['path'] = Config.BASE_DIR
with open(dynamic_yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)

model_weights = "yolov8n.pt"
resume_training = False
last_ckpt = os.path.join(Config.YOLO_PROJ_DIR, "train", "weights", "last.pt")

if os.path.exists(last_ckpt):
    print("Found existing YOLO checkpoint. Preparing to resume training...")
    model_weights = last_ckpt
    resume_training = True

model = YOLO(model_weights)

print(f"Starting YOLOv8 Training. Device: {Config.DEVICE}, Mixed Precision: {Config.MIXED_PRECISION}")
train_results = model.train(
    data=dynamic_yaml_path,
    epochs=Config.EPOCHS,
    batch=Config.BATCH_SIZE,
    imgsz=Config.IMG_SIZE,
    workers=Config.WORKERS,
    project=Config.YOLO_PROJ_DIR,
    name="train",
    exist_ok=True,
    resume=resume_training,
    amp=Config.MIXED_PRECISION,
    lr0=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY,
    val=True,
    save=True,
    save_period=1
)

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/output/phase_1_2/yolo_train/train/weights/best.pt")

model.train(
    data="./output/phase_1_2/dynamic_data.yaml",
    epochs=50,
    imgsz=416,
    batch=16,
    project="./output/phase_1_2/yolo_train",
    name="train_continue",
    exist_ok=True
)

In [ ]:
results_csv_path = os.path.join(Config.YOLO_PROJ_DIR, "train", "results.csv")

if os.path.exists(results_csv_path):
    df_results = pd.read_csv(results_csv_path)
    df_results.columns = df_results.columns.str.strip()
    
    epochs = df_results['epoch']
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, df_results['train/box_loss'], label='Train Box Loss', marker='o')
    plt.plot(epochs, df_results['val/box_loss'], label='Val Box Loss', marker='o')
    plt.title('Bounding Box Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, df_results['train/cls_loss'], label='Train Cls Loss', marker='o')
    plt.plot(epochs, df_results['val/cls_loss'], label='Val Cls Loss', marker='o')
    plt.title('Classification Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(Config.PLOT_DIR, "training_loss_curves.png"), dpi=300)
    plt.show()

    plt.close()
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, df_results['metrics/mAP50(B)'], label='mAP@0.5', marker='o', color='purple')
    plt.plot(epochs, df_results['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', marker='o', color='green')
    plt.plot(epochs, df_results['metrics/precision(B)'], label='Precision', linestyle='--', color='blue')
    plt.plot(epochs, df_results['metrics/recall(B)'], label='Recall', linestyle='--', color='red')
    plt.title('Validation Metrics (mAP, Precision, Recall)')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(Config.PLOT_DIR, "validation_metrics_curves.png"), dpi=300)
    plt.show()
    plt.close()

yolo_plots = glob.glob(os.path.join(Config.YOLO_PROJ_DIR, "train", "*.png"))
for plot_path in yolo_plots:
    filename = os.path.basename(plot_path)
    shutil.copy(plot_path, os.path.join(Config.PLOT_DIR, f"yolo_internal_{filename}"))

In [ ]:
best_model_path = os.path.join(Config.YOLO_PROJ_DIR, "train", "weights", "best.pt")
if not os.path.exists(best_model_path):
    best_model_path = last_ckpt

if os.path.exists(best_model_path):
    inference_model = YOLO(best_model_path)
    
    sample_images = random.sample(val_images, min(5, len(val_images)))
    
    for idx, img_path in enumerate(sample_images):
        img_name = os.path.basename(img_path)
        results = inference_model.predict(source=img_path, imgsz=Config.IMG_SIZE, conf=0.25, save=False)
        
        for r in results:
            img_arr = r.plot()
            img_rgb = cv2.cvtColor(img_arr, cv2.COLOR_BGR2RGB)
            
            plt.figure(figsize=(8, 8))
            plt.imshow(img_rgb)
            plt.title(f"Inference Result: {img_name}")
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(Config.PLOT_DIR, f"inference_sample_{idx+1}.png"), dpi=300)
            plt.show()
            plt.close()

In [ ]:
if os.path.exists(results_csv_path):
    best_epoch_idx = df_results['metrics/mAP50-95(B)'].idxmax()
    best_metrics = df_results.iloc[best_epoch_idx]
    
    final_metrics = {
        "Model Name": "YOLOv8-Pest-Detection",
        "Best Epoch": int(best_metrics['epoch']),
        "Best mAP@0.5": float(best_metrics['metrics/mAP50(B)']),
        "Best mAP@0.5:0.95": float(best_metrics['metrics/mAP50-95(B)']),
        "Best Precision": float(best_metrics['metrics/precision(B)']),
        "Best Recall": float(best_metrics['metrics/recall(B)']),
        "Total Training Epochs": Config.EPOCHS,
        "Batch Size": Config.BATCH_SIZE,
        "Image Size": Config.IMG_SIZE,
        "Total Classes": num_classes,
        "Class Names": class_names
    }
    
    with open(os.path.join(Config.OUT_DIR, "final_detection_metrics.json"), "w") as f:
        json.dump(final_metrics, f, indent=4)
        
    pd.DataFrame([final_metrics]).to_csv(os.path.join(Config.OUT_DIR, "final_detection_metrics.csv"), index=False)
    
    export_metadata = {
        "class_mapping": {int(k): str(v) for k, v in enumerate(class_names)},
        "img_size": Config.IMG_SIZE
    }
    with open(os.path.join(Config.OUT_DIR, "detection_metadata.json"), "w") as f:
        json.dump(export_metadata, f, indent=4)

In [ ]:
print("\n--- Starting Advanced Post-Training Evaluation ---")
if os.path.exists(best_model_path):
    val_model = YOLO(best_model_path)
    
    try:
        total_params = sum(p.numel() for p in val_model.model.parameters())
        model_info = val_model.info() 
        gflops = model_info[3] if isinstance(model_info, tuple) and len(model_info) > 3 else "Check Logs"
        
        complexity_info = {
            "Parameters": total_params,
            "GFLOPs": gflops
        }
        with open(os.path.join(Config.OUT_DIR, "model_complexity.json"), "w") as f:
            json.dump(complexity_info, f, indent=4)
        print(f"Model Complexity: {total_params} Params, GFLOPs: {gflops}")
    except Exception as e:
        print(f"Warning: Could not extract complete model complexity: {e}")

    print("Generating Detailed Validation Metrics (Confusion Matrix, PR Curve, etc.)...")
    val_metrics = val_model.val(data=dynamic_yaml_path, split='val', save_json=True, plots=True, project=Config.YOLO_PROJ_DIR, name='val_standard', exist_ok=True)
    
    val_plots = glob.glob(os.path.join(Config.YOLO_PROJ_DIR, "val_standard", "*.png"))
    for plot_path in val_plots:
        filename = os.path.basename(plot_path)
        shutil.copy(plot_path, os.path.join(Config.PLOT_DIR, f"val_{filename}"))

    print("Running Test-Time Augmentation (TTA) Evaluation...")
    tta_metrics = val_model.val(data=dynamic_yaml_path, split='val', augment=True, project=Config.YOLO_PROJ_DIR, name='val_tta', exist_ok=True)
    tta_summary = {
        "mAP50": float(tta_metrics.box.map50),
        "mAP50-95": float(tta_metrics.box.map)
    }
    with open(os.path.join(Config.OUT_DIR, "tta_metrics.json"), "w") as f:
        json.dump(tta_summary, f, indent=4)
    print(f"TTA Results - mAP50: {tta_summary['mAP50']:.4f}, mAP50-95: {tta_summary['mAP50-95']:.4f}")

    print("Benchmarking Inference Latency...")
    dummy_imgs = random.sample(val_images, min(100, len(val_images))) if val_images else []
    if dummy_imgs:
        start_time = time.time()
        for img_path in dummy_imgs:
            _ = val_model.predict(img_path, imgsz=Config.IMG_SIZE, verbose=False)
        total_time = time.time() - start_time
        latency_ms = (total_time / len(dummy_imgs)) * 1000
        fps = 1000 / latency_ms if latency_ms > 0 else 0
        
        benchmark_info = {
            "Average_Latency_ms": round(latency_ms, 2),
            "FPS": round(fps, 2),
            "Device": Config.DEVICE,
            "Image_Size": Config.IMG_SIZE
        }
        with open(os.path.join(Config.OUT_DIR, "inference_benchmark.json"), "w") as f:
            json.dump(benchmark_info, f, indent=4)
        print(f"Benchmark: Latency {latency_ms:.2f} ms | {fps:.2f} FPS")

    print("Exporting Models for Production Deployment...")
    export_dir = os.path.join(Config.OUT_DIR, "exports")
    os.makedirs(export_dir, exist_ok=True)
    
    try:
        val_model.export(format='onnx', imgsz=Config.IMG_SIZE)
        onnx_path = best_model_path.replace(".pt", ".onnx")
        if os.path.exists(onnx_path): shutil.copy(onnx_path, export_dir)
        print("ONNX Export Successful.")
    except Exception as e:
        print(f"ONNX export failed: {e}")
        
    try:
        val_model.export(format='torchscript', imgsz=Config.IMG_SIZE)
        ts_path = best_model_path.replace(".pt", ".torchscript")
        if os.path.exists(ts_path): shutil.copy(ts_path, export_dir)
        print("TorchScript Export Successful.")
    except Exception as e:
        print(f"TorchScript export failed: {e}")

    try:
        val_model.export(format='engine', dynamic=True, workspace=2, imgsz=Config.IMG_SIZE)
        engine_path = best_model_path.replace(".pt", ".engine")
        if os.path.exists(engine_path): shutil.copy(engine_path, export_dir)
        print("TensorRT Export Successful.")
    except Exception:
        print("TensorRT export skipped (requires specific environment dependencies).")

    print("Generating Error Analysis Visualizations (False Positives / False Negatives)...")
    error_analysis_samples = dummy_imgs[:5]
    for idx, img_path in enumerate(error_analysis_samples):
        label_path = img_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"
        img = cv2.imread(img_path)
        if img is None: continue
        
        h, w = img.shape[:2]
        true_boxes = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cid = int(parts[0])
                        xc, yc, bw, bh = map(float, parts[1:])
                        x1 = int((xc - bw/2) * w)
                        y1 = int((yc - bh/2) * h)
                        x2 = int((xc + bw/2) * w)
                        y2 = int((yc + bh/2) * h)
                        true_boxes.append([x1, y1, x2, y2, cid])
        
        img_true = img.copy()
        for box in true_boxes:
            cv2.rectangle(img_true, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
            cv2.putText(img_true, f"GT: {class_names[box[4]]}", (box[0], max(0, box[1]-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        res = val_model.predict(img_path, imgsz=Config.IMG_SIZE, conf=0.25, verbose=False)[0]
        img_pred = img.copy()
        if res.boxes:
            for box in res.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                conf = float(box.conf[0])
                cid = int(box.cls[0])
                cv2.rectangle(img_pred, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(img_pred, f"PR: {class_names[cid]} {conf:.2f}", (x1, max(0, y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
            
        fig, ax = plt.subplots(1, 2, figsize=(12, 6))
        ax[0].imshow(cv2.cvtColor(img_true, cv2.COLOR_BGR2RGB))
        ax[0].set_title("Ground Truth (Green)")
        ax[0].axis('off')
        
        ax[1].imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB))
        ax[1].set_title("Predictions (Red)")
        ax[1].axis('off')
        
        plt.tight_layout()
        plt.savefig(os.path.join(Config.PLOT_DIR, f"error_analysis_{idx+1}.png"), dpi=300)
        plt.show()
        plt.close()

print(f"\nPhase 1.2 Pipeline Execution Complete. All artifacts saved to {Config.OUT_DIR}")

In [ ]:
import os
import glob
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.grid": True,
    "grid.color": "#dddddd",
    "grid.linestyle": "--",
    "grid.linewidth": 0.6,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 100,
    "savefig.dpi": 300,
})

OUT_DIR = "output/phase_1_2/graphs"
os.makedirs(OUT_DIR, exist_ok=True)

def _find(*patterns):
    for p in patterns:
        matches = glob.glob(p, recursive=True)
        if matches:
            matches.sort(key=lambda x: os.path.getmtime(x), reverse=True)
            return matches[0]
    return None

def save_fig(fig, name):
    try:
        fig.tight_layout()
        plt.show()
        path = os.path.join(OUT_DIR, f"{name}.png")
        fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        print(f"✓ Figure saved: {path}")
    except Exception as e:
        print(f"✗ Could not save {name}: {e}")
        try:
            plt.close(fig)
        except Exception:
            pass


def safe_block(title):
    def deco(fn):
        def wrapper(*args, **kwargs):
            try:
                return fn(*args, **kwargs)
            except Exception as e:
                print(f" Skipped [{title}]: {e}")
        return wrapper
    return deco

def copy_asset(name, patterns):
    src = _find(*patterns)
    if src is None:
        print(f" Skipped [{name}]: file not found")
        return None
    try:
        dst = os.path.join(OUT_DIR, f"{name}.png")
        shutil.copy(src, dst)
        img = mpimg.imread(dst)
        fig, ax = plt.subplots(figsize=(9, 8))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(name.replace("_", " ").title())
        fig.tight_layout()
        plt.show()
        plt.close(fig)
        print(f" Figure saved: {dst}")
        return dst
    except Exception as e:
        print(f"Skipped [{name}]: {e}")
        return None

results_csv_path = _find(
    "**/results.csv", "runs/**/results.csv", "/kaggle/working/**/results.csv"
)
results_df = None
if results_csv_path:
    try:
        results_df = pd.read_csv(results_csv_path)
        results_df.columns = [c.strip() for c in results_df.columns]
        print(f"✓ Loaded results.csv from: {results_csv_path}")
    except Exception as e:
        print(f"⚠ Could not read results.csv: {e}")
else:
    print("⚠ results.csv not found.")

def col(*candidates):
    if results_df is None:
        return None
    for c in candidates:
        if c in results_df.columns:
            return results_df[c].values
    return None

epochs = None
if results_df is not None:
    if "epoch" in results_df.columns:
        epochs = results_df["epoch"].values
    else:
        epochs = np.arange(len(results_df))

train_box_loss = col("train/box_loss")
train_cls_loss = col("train/cls_loss")
train_dfl_loss = col("train/dfl_loss")
val_box_loss   = col("val/box_loss")
val_cls_loss   = col("val/cls_loss")
val_dfl_loss   = col("val/dfl_loss")
precision      = col("metrics/precision(B)", "metrics/precision")
recall         = col("metrics/recall(B)", "metrics/recall")
map50          = col("metrics/mAP50(B)", "metrics/mAP_0.5")
map5095        = col("metrics/mAP50-95(B)", "metrics/mAP_0.5:0.95")
lr0            = col("lr/pg0")
lr1            = col("lr/pg1")
lr2            = col("lr/pg2")

print("=" * 60)
print("TRAINING CURVES")
print("=" * 60)

@safe_block("Train Box Loss")
def _tc1():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, train_box_loss, color="#1f77b4", linewidth=1.8)
    ax.set_title("Train Box Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    save_fig(fig, "01_train_box_loss")
_tc1()

@safe_block("Train Classification Loss")
def _tc2():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, train_cls_loss, color="#ff7f0e", linewidth=1.8)
    ax.set_title("Train Classification Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    save_fig(fig, "02_train_cls_loss")
_tc2()

@safe_block("Train DFL Loss")
def _tc3():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, train_dfl_loss, color="#2ca02c", linewidth=1.8)
    ax.set_title("Train DFL Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    save_fig(fig, "03_train_dfl_loss")
_tc3()

@safe_block("Validation Box Loss")
def _tc4():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, val_box_loss, color="#d62728", linewidth=1.8)
    ax.set_title("Validation Box Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    save_fig(fig, "04_val_box_loss")
_tc4()

@safe_block("Validation Classification Loss")
def _tc5():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, val_cls_loss, color="#9467bd", linewidth=1.8)
    ax.set_title("Validation Classification Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    save_fig(fig, "05_val_cls_loss")
_tc5()

@safe_block("Validation DFL Loss")
def _tc6():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, val_dfl_loss, color="#8c564b", linewidth=1.8)
    ax.set_title("Validation DFL Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    save_fig(fig, "06_val_dfl_loss")
_tc6()

print("=" * 60)
print("METRICS")
print("=" * 60)

@safe_block("Precision vs Epoch")
def _m1():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, precision, color="#1f77b4", linewidth=1.8)
    ax.set_title("Precision vs Epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("Precision")
    save_fig(fig, "07_precision_vs_epoch")
_m1()

@safe_block("Recall vs Epoch")
def _m2():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, recall, color="#ff7f0e", linewidth=1.8)
    ax.set_title("Recall vs Epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("Recall")
    save_fig(fig, "08_recall_vs_epoch")
_m2()

@safe_block("mAP@50 vs Epoch")
def _m3():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, map50, color="#2ca02c", linewidth=1.8)
    ax.set_title("mAP@50 vs Epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("mAP@50")
    save_fig(fig, "09_map50_vs_epoch")
_m3()

@safe_block("mAP@50-95 vs Epoch")
def _m4():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, map5095, color="#d62728", linewidth=1.8)
    ax.set_title("mAP@50-95 vs Epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("mAP@50-95")
    save_fig(fig, "10_map5095_vs_epoch")
_m4()

print("=" * 60)
print("COMBINED FIGURES")
print("=" * 60)

@safe_block("Loss Comparison")
def _cf1():
    fig, ax = plt.subplots(figsize=(9, 6))
    series = [
        ("Train Box", train_box_loss, "#1f77b4"),
        ("Train Cls", train_cls_loss, "#ff7f0e"),
        ("Train DFL", train_dfl_loss, "#2ca02c"),
        ("Val Box", val_box_loss, "#d62728"),
        ("Val Cls", val_cls_loss, "#9467bd"),
        ("Val DFL", val_dfl_loss, "#8c564b"),
    ]
    plotted = False
    for label, data, color in series:
        if data is not None:
            ax.plot(epochs, data, label=label, color=color, linewidth=1.6)
            plotted = True
    if not plotted:
        raise ValueError("no loss columns available")
    ax.set_title("Loss Comparison"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend()
    save_fig(fig, "11_loss_comparison")
_cf1()

@safe_block("Metric Comparison")
def _cf2():
    fig, ax = plt.subplots(figsize=(9, 6))
    series = [
        ("Precision", precision, "#1f77b4"),
        ("Recall", recall, "#ff7f0e"),
        ("mAP@50", map50, "#2ca02c"),
        ("mAP@50-95", map5095, "#d62728"),
    ]
    plotted = False
    for label, data, color in series:
        if data is not None:
            ax.plot(epochs, data, label=label, color=color, linewidth=1.8)
            plotted = True
    if not plotted:
        raise ValueError("no metric columns available")
    ax.set_title("Metric Comparison"); ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
    ax.legend()
    save_fig(fig, "12_metric_comparison")
_cf2()

@safe_block("Precision-Recall Trend")
def _cf3():
    if precision is None or recall is None:
        raise ValueError("precision or recall missing")
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(recall, precision, c=epochs, cmap="viridis", s=30)
    ax.plot(recall, precision, color="gray", alpha=0.4, linewidth=1)
    ax.set_title("Precision-Recall Trend Across Epochs")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Epoch")
    save_fig(fig, "13_precision_recall_trend")
_cf3()

@safe_block("Learning Progress Dashboard")
def _cf4():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    try:
        if train_box_loss is not None:
            axes[0, 0].plot(epochs, train_box_loss, label="Train Box", color="#1f77b4")
        if val_box_loss is not None:
            axes[0, 0].plot(epochs, val_box_loss, label="Val Box", color="#d62728")
        axes[0, 0].set_title("Box Loss"); axes[0, 0].legend(); axes[0, 0].set_xlabel("Epoch")
    except Exception:
        pass
    try:
        if train_cls_loss is not None:
            axes[0, 1].plot(epochs, train_cls_loss, label="Train Cls", color="#ff7f0e")
        if val_cls_loss is not None:
            axes[0, 1].plot(epochs, val_cls_loss, label="Val Cls", color="#9467bd")
        axes[0, 1].set_title("Classification Loss"); axes[0, 1].legend(); axes[0, 1].set_xlabel("Epoch")
    except Exception:
        pass
    try:
        if precision is not None:
            axes[1, 0].plot(epochs, precision, label="Precision", color="#1f77b4")
        if recall is not None:
            axes[1, 0].plot(epochs, recall, label="Recall", color="#ff7f0e")
        axes[1, 0].set_title("Precision & Recall"); axes[1, 0].legend(); axes[1, 0].set_xlabel("Epoch")
    except Exception:
        pass
    try:
        if map50 is not None:
            axes[1, 1].plot(epochs, map50, label="mAP@50", color="#2ca02c")
        if map5095 is not None:
            axes[1, 1].plot(epochs, map5095, label="mAP@50-95", color="#d62728")
        axes[1, 1].set_title("mAP Progress"); axes[1, 1].legend(); axes[1, 1].set_xlabel("Epoch")
    except Exception:
        pass
    fig.suptitle("Learning Progress Dashboard", fontsize=15, fontweight="bold")
    save_fig(fig, "14_learning_progress_dashboard")
_cf4()

print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

copy_asset("15_confusion_matrix", ["**/confusion_matrix.png"])
copy_asset("16_confusion_matrix_normalized", ["**/confusion_matrix_normalized.png"])

print("=" * 60)
print("CLASS ANALYSIS")
print("=" * 60)

labels_csv_path = _find("**/labels.csv")
labels_df = None
if labels_csv_path:
    try:
        labels_df = pd.read_csv(labels_csv_path)
    except Exception:
        labels_df = None

class_names_glob = None
try:
    yaml_path = _find("**/*.yaml", "**/data.yaml")
    if yaml_path:
        import yaml
        with open(yaml_path, "r") as f:
            data_yaml = yaml.safe_load(f)
        names = data_yaml.get("names")
        if isinstance(names, dict):
            class_names_glob = [names[k] for k in sorted(names.keys())]
        elif isinstance(names, list):
            class_names_glob = names
except Exception:
    class_names_glob = None

@safe_block("Instances per Class")
def _ca1():
    labels_dir = _find("**/labels/train", "**/train/labels")
    if labels_dir is None:
        raise ValueError("labels directory not found")
    counts = {}
    for f in glob.glob(os.path.join(labels_dir, "*.txt")):
        with open(f, "r") as fh:
            for line in fh:
                parts = line.strip().split()
                if not parts:
                    continue
                cid = int(float(parts[0]))
                counts[cid] = counts.get(cid, 0) + 1
    if not counts:
        raise ValueError("no label instances found")
    ids = sorted(counts.keys())
    names = [class_names_glob[i] if class_names_glob and i < len(class_names_glob) else str(i) for i in ids]
    values = [counts[i] for i in ids]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(names, values, color="#1f77b4", edgecolor="black")
    ax.set_title("Instances per Class"); ax.set_xlabel("Class"); ax.set_ylabel("Instances")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    save_fig(fig, "17_instances_per_class")
_ca1()

copy_asset("18_pr_curve", ["**/PR_curve.png"])
copy_asset("19_f1_curve", ["**/F1_curve.png"])
copy_asset("20_p_curve", ["**/P_curve.png"])
copy_asset("21_r_curve", ["**/R_curve.png"])

print("=" * 60)
print("DATASET ANALYSIS")
print("=" * 60)

def _load_all_label_boxes():
    labels_dir = _find("**/labels/train", "**/train/labels")
    boxes = []
    class_ids = []
    obj_per_image = []
    if labels_dir is None:
        raise ValueError("labels directory not found")
    for f in glob.glob(os.path.join(labels_dir, "*.txt")):
        count = 0
        with open(f, "r") as fh:
            for line in fh:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cid, xc, yc, w, h = parts[:5]
                boxes.append((float(w), float(h)))
                class_ids.append(int(float(cid)))
                count += 1
        obj_per_image.append(count)
    if not boxes:
        raise ValueError("no bounding boxes found")
    return np.array(boxes), np.array(class_ids), np.array(obj_per_image)

@safe_block("Bounding Box Width/Height/Area/Aspect Ratio Distributions")
def _da1():
    boxes, _, _ = _load_all_label_boxes()
    widths, heights = boxes[:, 0], boxes[:, 1]
    areas = widths * heights
    aspect = widths / np.clip(heights, 1e-6, None)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes[0, 0].hist(widths, bins=30, color="#1f77b4", edgecolor="black")
    axes[0, 0].set_title("Bounding Box Width Distribution"); axes[0, 0].set_xlabel("Normalized Width")

    axes[0, 1].hist(heights, bins=30, color="#ff7f0e", edgecolor="black")
    axes[0, 1].set_title("Bounding Box Height Distribution"); axes[0, 1].set_xlabel("Normalized Height")

    axes[1, 0].hist(areas, bins=30, color="#2ca02c", edgecolor="black")
    axes[1, 0].set_title("Bounding Box Area Distribution"); axes[1, 0].set_xlabel("Normalized Area")

    axes[1, 1].hist(aspect, bins=30, color="#d62728", edgecolor="black")
    axes[1, 1].set_title("Aspect Ratio Distribution"); axes[1, 1].set_xlabel("Width / Height")

    save_fig(fig, "22_bbox_distributions")
_da1()

@safe_block("Object Count per Image")
def _da2():
    _, _, obj_per_image = _load_all_label_boxes()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(obj_per_image, bins=range(0, int(obj_per_image.max()) + 2), color="#9467bd", edgecolor="black")
    ax.set_title("Object Count per Image"); ax.set_xlabel("Objects per Image"); ax.set_ylabel("Frequency")
    save_fig(fig, "23_object_count_per_image")
_da2()

@safe_block("Train/Validation/Test Image Distribution")
def _da3():
    splits = {}
    for split_name, patterns in [
        ("Train", ["**/images/train", "**/train/images"]),
        ("Validation", ["**/images/val", "**/val/images", "**/images/valid", "**/valid/images"]),
        ("Test", ["**/images/test", "**/test/images"]),
    ]:
        d = _find(*patterns)
        if d:
            n = len([f for f in glob.glob(os.path.join(d, "*")) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
            if n > 0:
                splits[split_name] = n
    if not splits:
        raise ValueError("no split image folders found")
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(list(splits.keys()), list(splits.values()), color=["#1f77b4", "#ff7f0e", "#2ca02c"], edgecolor="black")
    ax.set_title("Train / Validation / Test Image Distribution"); ax.set_ylabel("Number of Images")
    save_fig(fig, "24_split_image_distribution")
_da3()

print("=" * 60)
print("YOLO VISUALS")
print("=" * 60)

copy_asset("25_label_distribution", ["**/labels.jpg", "**/labels.png"])
copy_asset("26_label_correlogram", ["**/labels_correlogram.jpg", "**/labels_correlogram.png"])

@safe_block("Sample Predictions")
def _yv1():
    pred_dir = _find("**/predict", "**/val_batch0_pred.jpg")
    img_path = _find("**/val_batch0_pred.jpg", "**/predict/*.jpg")
    if img_path is None:
        raise ValueError("no prediction sample images found")
    img = mpimg.imread(img_path)
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img)
    ax.set_title("Sample Predictions")
    ax.axis("off")
    save_fig(fig, "27_sample_predictions")
_yv1()

@safe_block("Mosaic Samples")
def _yv2():
    img_path = _find("**/train_batch0.jpg", "**/train_batch*.jpg")
    if img_path is None:
        raise ValueError("no mosaic sample images found")
    img = mpimg.imread(img_path)
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img)
    ax.set_title("Mosaic Training Samples")
    ax.axis("off")
    save_fig(fig, "28_mosaic_samples")
_yv2()

@safe_block("Prediction Confidence Histogram")
def _yv3():
    conf_scores = None
    for name in ["pred_confidences", "confidences", "conf_scores", "y_prob"]:
        if name in globals():
            conf_scores = globals()[name]
            break
    if conf_scores is None:
        raise ValueError("no prediction confidence data found")
    conf_scores = np.array(conf_scores).ravel()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(conf_scores, bins=30, color="#17becf", edgecolor="black")
    ax.set_title("Prediction Confidence Histogram")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Frequency")
    save_fig(fig, "29_prediction_confidence_histogram")
_yv3()

print("=" * 60)
print("OPTIONAL ANALYSIS")
print("=" * 60)

@safe_block("Best Epoch Identification")
def _oa1():
    if map5095 is None:
        raise ValueError("mAP50-95 column not found")
    best_idx = int(np.argmax(map5095))
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(epochs, map5095, color="#2ca02c", linewidth=1.8, label="mAP@50-95")
    ax.scatter([epochs[best_idx]], [map5095[best_idx]], color="red", zorder=5,
               label=f"Best Epoch: {epochs[best_idx]} ({map5095[best_idx]:.3f})")
    ax.set_title("Best Epoch Identification (mAP@50-95)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("mAP@50-95")
    ax.legend()
    save_fig(fig, "30_best_epoch_identification")
    print(f"Best epoch: {epochs[best_idx]} with mAP@50-95 = {map5095[best_idx]:.4f}")
_oa1()

def moving_average(data, window=5):
    data = np.array(data, dtype=float)
    if len(data) < window:
        window = max(1, len(data) // 2)
    if window <= 1:
        return data
    return np.convolve(data, np.ones(window) / window, mode="valid")

@safe_block("Moving Average of Losses")
def _oa2():
    fig, ax = plt.subplots(figsize=(9, 6))
    plotted = False
    for label, data, color in [
        ("Train Box (MA)", train_box_loss, "#1f77b4"),
        ("Train Cls (MA)", train_cls_loss, "#ff7f0e"),
        ("Train DFL (MA)", train_dfl_loss, "#2ca02c"),
    ]:
        if data is not None:
            ma = moving_average(data, window=5)
            ax.plot(range(len(ma)), ma, label=label, color=color, linewidth=1.8)
            plotted = True
    if not plotted:
        raise ValueError("no loss data available")
    ax.set_title("Moving Average of Training Losses")
    ax.set_xlabel("Epoch (windowed)"); ax.set_ylabel("Loss (smoothed)")
    ax.legend()
    save_fig(fig, "31_moving_average_losses")
_oa2()

@safe_block("Moving Average of mAP")
def _oa3():
    if map5095 is None and map50 is None:
        raise ValueError("no mAP data available")
    fig, ax = plt.subplots(figsize=(9, 6))
    if map50 is not None:
        ma50 = moving_average(map50, window=5)
        ax.plot(range(len(ma50)), ma50, label="mAP@50 (MA)", color="#2ca02c", linewidth=1.8)
    if map5095 is not None:
        ma5095 = moving_average(map5095, window=5)
        ax.plot(range(len(ma5095)), ma5095, label="mAP@50-95 (MA)", color="#d62728", linewidth=1.8)
    ax.set_title("Moving Average of mAP")
    ax.set_xlabel("Epoch (windowed)"); ax.set_ylabel("mAP (smoothed)")
    ax.legend()
    save_fig(fig, "32_moving_average_map")
_oa3()

@safe_block("Learning Rate Schedule")
def _oa4():
    if lr0 is None and lr1 is None and lr2 is None:
        raise ValueError("no learning rate columns found")
    fig, ax = plt.subplots(figsize=(9, 5))
    if lr0 is not None:
        ax.plot(epochs, lr0, label="lr/pg0", color="#1f77b4")
    if lr1 is not None:
        ax.plot(epochs, lr1, label="lr/pg1", color="#ff7f0e")
    if lr2 is not None:
        ax.plot(epochs, lr2, label="lr/pg2", color="#2ca02c")
    ax.set_title("Learning Rate Schedule")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Learning Rate")
    ax.set_yscale("log")
    ax.legend()
    save_fig(fig, "33_learning_rate_schedule")
_oa4()

print("=" * 60)
print("FINAL DASHBOARD")
print("=" * 60)

@safe_block("Final Summary Dashboard")
def _dash():
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 2)

    ax1 = fig.add_subplot(gs[0, 0])
    try:
        if train_box_loss is not None:
            ax1.plot(epochs, train_box_loss, label="Train Box", color="#1f77b4")
        if val_box_loss is not None:
            ax1.plot(epochs, val_box_loss, label="Val Box", color="#d62728")
        ax1.set_title("Box Loss"); ax1.legend(); ax1.set_xlabel("Epoch")
    except Exception:
        pass

    ax2 = fig.add_subplot(gs[0, 1])
    try:
        if precision is not None:
            ax2.plot(epochs, precision, label="Precision", color="#1f77b4")
        if recall is not None:
            ax2.plot(epochs, recall, label="Recall", color="#ff7f0e")
        ax2.set_title("Precision & Recall"); ax2.legend(); ax2.set_xlabel("Epoch")
    except Exception:
        pass

    ax3 = fig.add_subplot(gs[1, 0])
    try:
        if map50 is not None:
            ax3.plot(epochs, map50, label="mAP@50", color="#2ca02c")
        if map5095 is not None:
            ax3.plot(epochs, map5095, label="mAP@50-95", color="#d62728")
        ax3.set_title("mAP Progress"); ax3.legend(); ax3.set_xlabel("Epoch")
    except Exception:
        pass

    ax4 = fig.add_subplot(gs[1, 1])
    try:
        cm_path = os.path.join(OUT_DIR, "16_confusion_matrix_normalized.png")
        if not os.path.exists(cm_path):
            cm_path = os.path.join(OUT_DIR, "15_confusion_matrix.png")
        img = mpimg.imread(cm_path)
        ax4.imshow(img)
        ax4.set_title("Confusion Matrix")
        ax4.axis("off")
    except Exception:
        ax4.text(0.5, 0.5, "Confusion Matrix N/A", ha="center", va="center")
        ax4.axis("off")

    ax5 = fig.add_subplot(gs[2, :])
    try:
        img_path = os.path.join(OUT_DIR, "27_sample_predictions.png")
        img = mpimg.imread(img_path)
        ax5.imshow(img)
        ax5.set_title("Sample Predictions")
        ax5.axis("off")
    except Exception:
        ax5.text(0.5, 0.5, "Sample Predictions N/A", ha="center", va="center")
        ax5.axis("off")

    fig.suptitle("YOLOv8 Rice Pest Detection - Final Results Dashboard", fontsize=17, fontweight="bold")
    save_fig(fig, "34_final_dashboard")
_dash()

print("=" * 60)
print("ALL AVAILABLE VISUALIZATIONS EXPORTED TO:", OUT_DIR)
print("=" * 60)

In [ ]:
import os
import glob
from IPython.display import display, Image, Markdown

OUT_DIR = "output/phase_1_2/graphs"

png_files = sorted(glob.glob(os.path.join(OUT_DIR, "*.png")))

if not png_files:
    print(" No saved graphs found in:", OUT_DIR)
else:
    print(f"Displaying {len(png_files)} graphs from: {OUT_DIR}\n")
    for path in png_files:
        name = os.path.splitext(os.path.basename(path))[0]
        title = name.split("_", 1)[-1].replace("_", " ").title() if "_" in name else name
        display(Markdown(f"### {title}"))
        display(Image(filename=path))

In [ ]:
# ============================================================
# CELL XX : Create Final Submission ZIP
# ============================================================

import os
import zipfile
from pathlib import Path

# Output ZIP
ZIP_NAME = "/kaggle/working/Rice_Pest_Detection_Final_Submission.zip"

# Directories to include
SEARCH_DIRS = [
    "/kaggle/working",
    "./output",
]

# File extensions to include
INCLUDE_EXTENSIONS = {
    ".pt",
    ".pth",
    ".onnx",
    ".engine",
    ".torchscript",
    ".json",
    ".yaml",
    ".yml",
    ".csv",
    ".txt",
    ".log",
    ".png",
    ".jpg",
    ".jpeg",
    ".pdf",
    ".xlsx",
    ".xml"
}

files_added = 0

with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zipf:

    added = set()

    for directory in SEARCH_DIRS:

        if not os.path.exists(directory):
            continue

        for root, dirs, files in os.walk(directory):

            # Skip cache folders
            dirs[:] = [
                d for d in dirs
                if d not in {
                    "__pycache__",
                    ".ipynb_checkpoints",
                    ".cache"
                }
            ]

            for file in files:

                path = os.path.join(root, file)

                if Path(file).suffix.lower() not in INCLUDE_EXTENSIONS:
                    continue

                real_path = os.path.abspath(path)

                if real_path in added:
                    continue

                added.add(real_path)

                arcname = os.path.relpath(real_path, "/kaggle/working")

                zipf.write(real_path, arcname)

                files_added += 1

print("=" * 60)
print("ZIP Successfully Created")
print("=" * 60)
print("Location :", ZIP_NAME)
print("Files Added :", files_added)
print("=" * 60)